# Surveillance Frame Dataset Preparation

This notebook creates a **frame-level dataset** for the Conference Speech Offline Video Analyser.

**Pipeline:**
1. Download surveillance videos (YouTube + synthetic generation)
2. Normalize all videos to 720p H.264 mp4
3. Extract frames at 1 FPS
4. Split into two folders:
   - `normal/` -- frames from clean surveillance footage
   - `testing/` -- anomaly frames with subfolders: `blocked/`, `blur/`, `moved/`, `fire/`, `smoke/`, `person/`, `vehicle/`
5. Zip and download

**Output:** Two zip files
- `surveillance_videos.zip` -- normalized videos for `ConferenceSpeech/videos/`
- `surveillance_frames_dataset.zip` -- `normal/` + `testing/` frame folders

---
## Cell 1: Install Tools

In [ ]:
!pip install -q yt-dlp gdown opencv-python-headless tqdm
!which ffmpeg || (echo 'Installing ffmpeg...' && apt-get install -y -qq ffmpeg > /dev/null 2>&1)

import subprocess
for tool in ['yt-dlp', 'ffmpeg']:
    r = subprocess.run([tool, '--version'], capture_output=True, text=True)
    ver = r.stdout.strip().split('\n')[0] if r.returncode == 0 else 'NOT FOUND'
    print(f'{tool}: {ver}')

print('\nAll tools ready.')

---
## Cell 2: Define Video Sources

Curated list of public surveillance clips organized by category.

**To add your own:** Edit the `VIDEO_SOURCES` dict below. Each entry needs `url`, `filename`, and `max_duration`.

In [ ]:
import os
import json
import subprocess
import shutil
from pathlib import Path
from tqdm.notebook import tqdm

# === Directories ===
RAW_DIR      = Path('/content/raw_videos')
VIDEO_DIR    = Path('/content/videos')
FRAMES_DIR   = Path('/content/frames_tmp')
DATASET_DIR  = Path('/content/dataset')
NORMAL_DIR   = DATASET_DIR / 'normal'
TESTING_DIR  = DATASET_DIR / 'testing'

for d in [RAW_DIR, VIDEO_DIR, FRAMES_DIR, DATASET_DIR, NORMAL_DIR, TESTING_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# === Video Sources ===
# Categories: normal, fire_or_smoke, person_detected, vehicle_detected
# (blocked, blur, moved are generated synthetically in Cell 4)

VIDEO_SOURCES = {
    'normal': [
        {'url': 'https://www.youtube.com/watch?v=MNn9qKG2UFI', 'filename': 'normal_street_01.mp4',   'max_duration': 50, 'description': 'Street surveillance daytime'},
        {'url': 'https://www.youtube.com/watch?v=wqctLW0Hb_0', 'filename': 'normal_parking_01.mp4',  'max_duration': 50, 'description': 'Parking lot CCTV'},
        {'url': 'https://www.youtube.com/watch?v=1EiC9bvVGnk', 'filename': 'normal_indoor_01.mp4',   'max_duration': 50, 'description': 'Indoor hallway camera'},
        {'url': 'https://www.youtube.com/watch?v=Jpi4VQ4wGyI', 'filename': 'normal_night_01.mp4',    'max_duration': 50, 'description': 'Night time CCTV footage'},
        {'url': 'https://www.youtube.com/watch?v=7bfKREAoYSE', 'filename': 'normal_road_01.mp4',     'max_duration': 50, 'description': 'Road surveillance camera'},
    ],
    'fire_or_smoke': [
        {'url': 'https://www.youtube.com/watch?v=0-EkWMRFBjg', 'filename': 'fire_cctv_01.mp4',       'max_duration': 50, 'description': 'Fire caught on CCTV'},
        {'url': 'https://www.youtube.com/watch?v=cGHSjLOuXY0', 'filename': 'smoke_detect_01.mp4',    'max_duration': 50, 'description': 'Smoke visible on camera'},
        {'url': 'https://www.youtube.com/watch?v=GFBfOhFRMIo', 'filename': 'fire_indoor_01.mp4',     'max_duration': 50, 'description': 'Indoor fire on surveillance'},
    ],
    'person_detected': [
        {'url': 'https://www.youtube.com/watch?v=Gm2x6CVFpbI', 'filename': 'person_walk_01.mp4',     'max_duration': 50, 'description': 'Person walking in view'},
        {'url': 'https://www.youtube.com/watch?v=aUdKzb4LGJI', 'filename': 'person_hallway_01.mp4',  'max_duration': 50, 'description': 'People in hallway CCTV'},
        {'url': 'https://www.youtube.com/watch?v=Y1jTpCe1jLo', 'filename': 'person_loiter_01.mp4',   'max_duration': 50, 'description': 'Person loitering on camera'},
    ],
    'vehicle_detected': [
        {'url': 'https://www.youtube.com/watch?v=PJ5xXXcfuTc', 'filename': 'vehicle_traffic_01.mp4', 'max_duration': 50, 'description': 'Traffic camera footage'},
        {'url': 'https://www.youtube.com/watch?v=jjlBnrzSGjc', 'filename': 'vehicle_parking_01.mp4', 'max_duration': 50, 'description': 'Vehicle in parking lot'},
        {'url': 'https://www.youtube.com/watch?v=KBsqQez-O4w', 'filename': 'vehicle_road_01.mp4',    'max_duration': 50, 'description': 'Cars on road surveillance'},
    ],
}

total = sum(len(v) for v in VIDEO_SOURCES.values())
print(f'Defined {total} downloadable videos + 6 synthetic (blocked/blur/moved) = {total + 6} total')
print()
for cat, vids in VIDEO_SOURCES.items():
    print(f'  {cat}: {len(vids)} videos')
print(f'  blocked (synthetic): 2 videos')
print(f'  blur (synthetic): 2 videos')
print(f'  moved (synthetic): 2 videos')

---
## Cell 3: Download and Normalize Videos

Downloads each video, then normalizes to 720p H.264 mp4 (max 60s).

In [ ]:
def is_youtube(url):
    return 'youtube.com' in url or 'youtu.be' in url


def download_and_normalize(url, raw_path, norm_path, max_duration=60):
    """Download a video and normalize to 720p H.264 mp4."""
    # Step 1: Download raw
    if not os.path.exists(raw_path) or os.path.getsize(raw_path) < 1000:
        if is_youtube(url):
            cmd = [
                'yt-dlp', '--no-warnings',
                '-f', 'best[height<=720][ext=mp4]/best[height<=720]/best[ext=mp4]/best',
                '--no-playlist', '-o', raw_path, url,
            ]
        else:
            cmd = ['wget', '-q', '-O', raw_path, url]

        result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
        if result.returncode != 0:
            print(f'    Download error: {result.stderr[:150]}')
            return False

    if not os.path.exists(raw_path) or os.path.getsize(raw_path) < 1000:
        return False

    # Step 2: Normalize to 720p H.264, trim to max_duration
    norm_cmd = [
        'ffmpeg', '-y', '-i', raw_path,
        '-t', str(max_duration),
        '-vf', 'scale=-2:720',
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
        '-c:a', 'aac', '-b:a', '128k',
        '-movflags', '+faststart',
        norm_path,
    ]
    result = subprocess.run(norm_cmd, capture_output=True, text=True, timeout=240)
    return os.path.exists(norm_path) and os.path.getsize(norm_path) > 1000


# Download all videos
download_results = []
idx = 0
total = sum(len(v) for v in VIDEO_SOURCES.values())

for category, videos in VIDEO_SOURCES.items():
    print(f'\n=== {category.upper()} ===')
    for vid in videos:
        idx += 1
        raw_path = str(RAW_DIR / vid['filename'])
        norm_path = str(VIDEO_DIR / vid['filename'])

        if os.path.exists(norm_path) and os.path.getsize(norm_path) > 1000:
            print(f'  [{idx}/{total}] {vid["filename"]} -- already normalized, skipping')
            download_results.append({'file': vid['filename'], 'category': category, 'status': 'exists'})
            continue

        print(f'  [{idx}/{total}] {vid["filename"]} -- downloading + normalizing...')
        ok = download_and_normalize(vid['url'], raw_path, norm_path, vid.get('max_duration', 60))

        if ok:
            size_mb = os.path.getsize(norm_path) / (1024 * 1024)
            print(f'           Done ({size_mb:.1f} MB)')
        else:
            print(f'           FAILED')

        download_results.append({'file': vid['filename'], 'category': category, 'status': 'ok' if ok else 'FAILED'})

succeeded = sum(1 for r in download_results if r['status'] in ('ok', 'exists'))
print(f'\n=== Download complete: {succeeded}/{total} succeeded ===')

---
## Cell 4: Generate Synthetic Anomaly Videos

Creates **blocked**, **blur**, and **camera moved** videos from normal footage.
These anomaly types are difficult to find as public recordings, so we simulate them.

In [ ]:
import cv2
import numpy as np

def find_source_video():
    """Find a successfully downloaded normal video to use as the base."""
    for f in sorted(VIDEO_DIR.glob('normal_*.mp4')):
        if f.stat().st_size > 10000:
            return str(f)
    for f in sorted(VIDEO_DIR.glob('*.mp4')):
        if f.stat().st_size > 10000:
            return str(f)
    return None


def _open_video(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    return cap, fps, w, h


def _read_or_loop(cap):
    ret, frame = cap.read()
    if not ret:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ret, frame = cap.read()
    return ret, frame


def generate_blocked(source, out_path, duration_s=35):
    """5s normal, then lens covered with dark overlay + noise."""
    cap, fps, w, h = _open_video(source)
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    normal_f = int(5 * fps)
    total_f = int(duration_s * fps)
    color = (22, 18, 12)

    for i in range(total_f):
        ret, frame = _read_or_loop(cap)
        if not ret: break
        if i >= normal_f:
            alpha = min(1.0, (i - normal_f) / (fps * 2))
            overlay = np.full_like(frame, color, dtype=np.uint8)
            noise = np.random.randint(-5, 6, frame.shape, dtype=np.int16)
            overlay = np.clip(overlay.astype(np.int16) + noise, 0, 255).astype(np.uint8)
            frame = cv2.addWeighted(frame, 1.0 - alpha, overlay, alpha, 0)
        writer.write(frame)

    cap.release(); writer.release()
    print(f'  Created: {os.path.basename(out_path)}')


def generate_blur(source, out_path, duration_s=35):
    """5s normal, then progressive Gaussian blur (defocus)."""
    cap, fps, w, h = _open_video(source)
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    normal_f = int(5 * fps)
    total_f = int(duration_s * fps)

    for i in range(total_f):
        ret, frame = _read_or_loop(cap)
        if not ret: break
        if i >= normal_f:
            progress = min(1.0, (i - normal_f) / (fps * 3))
            ksize = int(3 + progress * 44)
            if ksize % 2 == 0: ksize += 1
            frame = cv2.GaussianBlur(frame, (ksize, ksize), 0)
        writer.write(frame)

    cap.release(); writer.release()
    print(f'  Created: {os.path.basename(out_path)}')


def generate_moved(source, out_path, duration_s=35):
    """8s normal, then camera shifts position over 2s."""
    cap, fps, w, h = _open_video(source)
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    normal_f = int(8 * fps)
    shift_f = int(2 * fps)
    total_f = int(duration_s * fps)
    max_dx = int(w * 0.15)
    max_dy = int(h * 0.10)

    for i in range(total_f):
        ret, frame = _read_or_loop(cap)
        if not ret: break
        if i >= normal_f:
            prog = min(1.0, (i - normal_f) / shift_f)
            dx, dy = int(max_dx * prog), int(max_dy * prog)
            M = np.float32([[1, 0, dx], [0, 1, dy]])
            frame = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        writer.write(frame)

    cap.release(); writer.release()
    print(f'  Created: {os.path.basename(out_path)}')


# Generate all synthetic videos
source = find_source_video()
if source:
    print(f'Source video: {os.path.basename(source)}\n')

    print('--- BLOCKED ---')
    generate_blocked(source, str(VIDEO_DIR / 'blocked_lens_01.mp4'), 35)
    generate_blocked(source, str(VIDEO_DIR / 'blocked_tape_02.mp4'), 30)

    print('\n--- BLUR ---')
    generate_blur(source, str(VIDEO_DIR / 'blur_defocus_01.mp4'), 35)
    generate_blur(source, str(VIDEO_DIR / 'blur_soft_02.mp4'), 30)

    print('\n--- CAMERA MOVED ---')
    generate_moved(source, str(VIDEO_DIR / 'moved_shift_01.mp4'), 35)
    generate_moved(source, str(VIDEO_DIR / 'moved_pan_02.mp4'), 30)

    print(f'\nSynthetic generation complete. Total videos in folder: {len(list(VIDEO_DIR.glob("*.mp4")))}')
else:
    print('ERROR: No source video found. Run Cell 3 first.')

---
## Cell 5: Extract Frames (1 FPS)

Extracts 1 frame per second from each video and saves as JPG.

In [ ]:
import cv2
import os
from pathlib import Path
from tqdm.notebook import tqdm

EXTRACT_FPS = 1  # frames per second to extract

def extract_frames(video_path, output_folder, target_fps=1):
    """Extract frames at target_fps from a video. Returns count of frames extracted."""
    cap = cv2.VideoCapture(str(video_path))
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_interval = max(1, int(video_fps / target_fps))

    os.makedirs(output_folder, exist_ok=True)
    video_name = Path(video_path).stem
    count = 0
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            out_path = os.path.join(output_folder, f'{video_name}_{count:04d}.jpg')
            cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
            count += 1
        frame_idx += 1

    cap.release()
    return count


# Clean old frames
if FRAMES_DIR.exists():
    shutil.rmtree(str(FRAMES_DIR))
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# Extract from all videos
video_files = sorted(VIDEO_DIR.glob('*.mp4'))
print(f'Extracting frames at {EXTRACT_FPS} FPS from {len(video_files)} videos...\n')

frame_counts = {}
for vf in tqdm(video_files, desc='Extracting'):
    out_folder = str(FRAMES_DIR / vf.stem)
    n = extract_frames(str(vf), out_folder, target_fps=EXTRACT_FPS)
    frame_counts[vf.stem] = n
    print(f'  {vf.name}: {n} frames')

total_frames = sum(frame_counts.values())
print(f'\nTotal frames extracted: {total_frames}')

---
## Cell 6: Organize into normal/ and testing/

Moves extracted frames into the final dataset structure:
- `dataset/normal/` -- frames from normal footage
- `dataset/testing/blocked/` -- blocked camera frames
- `dataset/testing/blur/` -- blurry camera frames
- `dataset/testing/moved/` -- camera shifted frames
- `dataset/testing/fire/` -- fire detected frames
- `dataset/testing/smoke/` -- smoke detected frames
- `dataset/testing/person/` -- person detected frames
- `dataset/testing/vehicle/` -- vehicle detected frames

In [ ]:
import shutil
from pathlib import Path

# Category mapping: filename prefix -> (folder, subfolder)
# 'normal' goes to dataset/normal/
# everything else goes to dataset/testing/<subfolder>/
CATEGORY_MAP = {
    'normal':   ('normal', None),
    'blocked':  ('testing', 'blocked'),
    'blur':     ('testing', 'blur'),
    'moved':    ('testing', 'moved'),
    'fire':     ('testing', 'fire'),
    'smoke':    ('testing', 'smoke'),
    'person':   ('testing', 'person'),
    'vehicle':  ('testing', 'vehicle'),
}

def classify_video(video_stem):
    """Determine category from the video filename."""
    name = video_stem.lower()
    for prefix, mapping in CATEGORY_MAP.items():
        if name.startswith(prefix):
            return mapping
    return ('normal', None)  # default to normal


# Clean old dataset
for d in [NORMAL_DIR, TESTING_DIR]:
    if d.exists():
        shutil.rmtree(str(d))
    d.mkdir(parents=True, exist_ok=True)

# Create testing subfolders
for sub in ['blocked', 'blur', 'moved', 'fire', 'smoke', 'person', 'vehicle']:
    (TESTING_DIR / sub).mkdir(exist_ok=True)

# Move frames
counts = {'normal': 0}
for sub in ['blocked', 'blur', 'moved', 'fire', 'smoke', 'person', 'vehicle']:
    counts[sub] = 0

frame_dirs = sorted(FRAMES_DIR.iterdir())
for fdir in frame_dirs:
    if not fdir.is_dir():
        continue

    folder_type, subfolder = classify_video(fdir.name)

    if folder_type == 'normal':
        dest = NORMAL_DIR
        key = 'normal'
    else:
        dest = TESTING_DIR / subfolder
        key = subfolder

    for jpg in sorted(fdir.glob('*.jpg')):
        shutil.copy2(str(jpg), str(dest / jpg.name))
        counts[key] += 1

# Print summary
normal_total = counts['normal']
testing_total = sum(v for k, v in counts.items() if k != 'normal')

print('=== Dataset Organization Complete ===\n')
print(f'normal/: {normal_total} frames')
print(f'testing/: {testing_total} frames')
for sub in ['blocked', 'blur', 'moved', 'fire', 'smoke', 'person', 'vehicle']:
    if counts[sub] > 0:
        print(f'  testing/{sub}/: {counts[sub]} frames')
print(f'\nTotal: {normal_total + testing_total} frames')

---
## Cell 7: Preview and Stats

Shows sample frames from each category and a summary table.

In [ ]:
import cv2
import base64
import os
from pathlib import Path
from IPython.display import display, HTML

def get_sample_frames(folder, n=4):
    """Get n evenly-spaced sample frame paths from a folder."""
    jpgs = sorted(Path(folder).glob('*.jpg'))
    if not jpgs:
        return []
    step = max(1, len(jpgs) // n)
    return [jpgs[i] for i in range(0, len(jpgs), step)][:n]


def frame_to_b64(path, size=(180, 135)):
    img = cv2.imread(str(path))
    if img is None:
        return None
    img = cv2.resize(img, size)
    _, buf = cv2.imencode('.jpg', img)
    return base64.b64encode(buf).decode('ascii')


# Build HTML preview
html = '<h2>Dataset Preview</h2>'

# Normal section
html += '<h3>normal/ folder</h3>'
html += '<div style="display:flex;flex-wrap:wrap;gap:8px;">'
for fp in get_sample_frames(NORMAL_DIR, 6):
    b64 = frame_to_b64(fp)
    if b64:
        html += (f'<div style="text-align:center;border:1px solid #4a4;padding:3px;border-radius:6px;">' 
                 f'<img src="data:image/jpeg;base64,{b64}" style="border-radius:4px;"/><br/>'
                 f'<small>{fp.name}</small></div>')
html += '</div>'

# Testing sections
for sub in ['blocked', 'blur', 'moved', 'fire', 'smoke', 'person', 'vehicle']:
    sub_dir = TESTING_DIR / sub
    n_files = len(list(sub_dir.glob('*.jpg')))
    if n_files == 0:
        continue
    html += f'<h3>testing/{sub}/ ({n_files} frames)</h3>'
    html += '<div style="display:flex;flex-wrap:wrap;gap:8px;">'
    for fp in get_sample_frames(sub_dir, 4):
        b64 = frame_to_b64(fp)
        if b64:
            html += (f'<div style="text-align:center;border:1px solid #c44;padding:3px;border-radius:6px;">'
                     f'<img src="data:image/jpeg;base64,{b64}" style="border-radius:4px;"/><br/>'
                     f'<small>{fp.name}</small></div>')
    html += '</div>'

display(HTML(html))

# Stats summary
print('\n=== Dataset Statistics ===')
normal_count = len(list(NORMAL_DIR.glob('*.jpg')))
normal_size = sum(f.stat().st_size for f in NORMAL_DIR.glob('*.jpg')) / (1024 * 1024)
print(f'normal/:   {normal_count:>5} frames  ({normal_size:.1f} MB)')

testing_total = 0
testing_size = 0
for sub in ['blocked', 'blur', 'moved', 'fire', 'smoke', 'person', 'vehicle']:
    sub_dir = TESTING_DIR / sub
    n = len(list(sub_dir.glob('*.jpg')))
    s = sum(f.stat().st_size for f in sub_dir.glob('*.jpg')) / (1024 * 1024)
    if n > 0:
        print(f'testing/{sub + "/":.<10s} {n:>4} frames  ({s:.1f} MB)')
    testing_total += n
    testing_size += s

print(f'\nTotal: {normal_count + testing_total} frames ({normal_size + testing_size:.1f} MB)')
print(f'  Normal:  {normal_count}')
print(f'  Testing: {testing_total}')

---
## Cell 8: Zip and Download

Creates two zip files:
1. **`surveillance_videos.zip`** -- normalized videos, unzip into `ConferenceSpeech/videos/`
2. **`surveillance_frames_dataset.zip`** -- `normal/` + `testing/` frame folders for analysis

Download both to your local machine.

In [ ]:
import shutil
import json
from google.colab import files

# --- Create manifest ---
manifest = {
    'normal': {'count': len(list(NORMAL_DIR.glob('*.jpg'))), 'folder': 'normal/'},
    'testing': {},
}
for sub in ['blocked', 'blur', 'moved', 'fire', 'smoke', 'person', 'vehicle']:
    n = len(list((TESTING_DIR / sub).glob('*.jpg')))
    if n > 0:
        manifest['testing'][sub] = {'count': n, 'folder': f'testing/{sub}/'}

manifest_path = DATASET_DIR / 'dataset_manifest.json'
with open(str(manifest_path), 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Manifest saved: {manifest_path}')

# --- Zip 1: Videos ---
vid_zip = '/content/surveillance_videos'
if os.path.exists(vid_zip + '.zip'):
    os.remove(vid_zip + '.zip')
shutil.make_archive(vid_zip, 'zip', str(VIDEO_DIR))
vid_zip_size = os.path.getsize(vid_zip + '.zip') / (1024 * 1024)
print(f'Videos zip: {vid_zip_size:.1f} MB')

# --- Zip 2: Frame Dataset ---
ds_zip = '/content/surveillance_frames_dataset'
if os.path.exists(ds_zip + '.zip'):
    os.remove(ds_zip + '.zip')
shutil.make_archive(ds_zip, 'zip', str(DATASET_DIR))
ds_zip_size = os.path.getsize(ds_zip + '.zip') / (1024 * 1024)
print(f'Frames zip: {ds_zip_size:.1f} MB')

print()
print('=' * 50)
print('DOWNLOAD INSTRUCTIONS')
print('=' * 50)
print()
print('1. surveillance_videos.zip')
print('   -> Unzip into ConferenceSpeech/videos/')
print('   -> Run: python main.py')
print('   -> Click "Start Processing"')
print()
print('2. surveillance_frames_dataset.zip')
print('   -> Contains normal/ and testing/ frame folders')
print('   -> Use for frame-level analysis and testing')
print()

# Download videos zip
print('Downloading surveillance_videos.zip ...')
files.download(vid_zip + '.zip')

# Download frames zip
print('Downloading surveillance_frames_dataset.zip ...')
files.download(ds_zip + '.zip')

---
## Cell 9 (Optional): Upload to Google Drive

Saves both zips to your Google Drive instead of browser download.

In [ ]:
# Uncomment the lines below to upload to Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
#
# dest = '/content/drive/MyDrive/'
# shutil.copy(vid_zip + '.zip', dest + 'surveillance_videos.zip')
# shutil.copy(ds_zip + '.zip', dest + 'surveillance_frames_dataset.zip')
# print(f'Uploaded both zips to Google Drive: {dest}')